# Temporary External AMR SMATCH

Notebook sementara untuk menghitung pairwise SMATCH dari satu file yang berisi beberapa AMR. Notebook ini tidak mengubah data maupun script proyek.

In [ ]:
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
import penman
from IPython.display import display

# Semua path ditulis absolut; ganti langsung jika file dipindahkan.
INPUT_FILE = Path(r'C:\Users\LEGION\.codex\attachments\a09f07c1-34ad-4f8e-87a7-fff0a736efd6\pasted-text.txt')
BUILDER_PATH = Path(r'D:\Github\generate_amr\scripts\liputan6\build_smatch_adjacency.py')

if not BUILDER_PATH.is_file():
    raise FileNotFoundError(f'Builder script tidak ditemukan: {BUILDER_PATH}')

spec = importlib.util.spec_from_file_location('build_smatch_adjacency', BUILDER_PATH)
builder = importlib.util.module_from_spec(spec)
spec.loader.exec_module(builder)

print(f'Input: {INPUT_FILE}')

In [ ]:
amr_text = INPUT_FILE.read_text(encoding='utf-8')
first_graph_position = amr_text.find('# ::snt')
if first_graph_position == -1:
    raise ValueError("Tidak menemukan metadata '# ::snt' dalam input.")

graphs = penman.loads(amr_text[first_graph_position:])
if not graphs:
    raise ValueError('Tidak ada AMR yang berhasil dibaca.')

sentences = []
for position, graph in enumerate(graphs):
    raw_id = graph.metadata.get('id', position)
    try:
        sentence_id = int(raw_id)
    except (TypeError, ValueError):
        sentence_id = raw_id
    sentences.append({
        'sent_idx': sentence_id,
        'sentence': graph.metadata.get('snt', ''),
        'amr': builder.graph_to_oneline(graph),
    })

sentence_table = pd.DataFrame([
    {'sent_idx': item['sent_idx'], 'sentence': item['sentence']}
    for item in sentences
])
print(f'Jumlah AMR: {len(sentences)}')
display(sentence_table)

In [ ]:
pairs = []
for i, first in enumerate(sentences):
    for second in sentences[i + 1:]:
        pairs.append({
            'sentence_a_idx': first['sent_idx'],
            'sentence_a': first['sentence'],
            'sentence_b_idx': second['sent_idx'],
            'sentence_b': second['sentence'],
            'smatch_score': builder.smatch_f_score(first['amr'], second['amr']),
        })

pairwise_scores = (
    pd.DataFrame(pairs)
    .sort_values('smatch_score', ascending=False)
    .reset_index(drop=True)
)
display(pairwise_scores.style.format({'smatch_score': '{:.4f}'}))

In [ ]:
amr_strings = [item['amr'] for item in sentences]
smatch_matrix = builder.compute_smatch_adjacency(amr_strings)
sentence_ids = [item['sent_idx'] for item in sentences]
matrix_table = pd.DataFrame(smatch_matrix, index=sentence_ids, columns=sentence_ids)
display(matrix_table.style.format('{:.4f}'))

## Detail satu pasangan

Ubah kedua indeks berikut untuk melihat AMR, alignment, dan matching triples.

In [ ]:
FIRST_SENTENCE_IDX = 0
SECOND_SENTENCE_IDX = 5

sentence_by_idx = {item['sent_idx']: item for item in sentences}
if FIRST_SENTENCE_IDX not in sentence_by_idx or SECOND_SENTENCE_IDX not in sentence_by_idx:
    raise ValueError('Kedua indeks harus tersedia pada sentence table.')
if FIRST_SENTENCE_IDX == SECOND_SENTENCE_IDX:
    raise ValueError('Pilih dua indeks yang berbeda.')

first_selected = sentence_by_idx[FIRST_SENTENCE_IDX]
second_selected = sentence_by_idx[SECOND_SENTENCE_IDX]
details = builder.smatch_details(first_selected['amr'], second_selected['amr'])

print(f"Sentence {FIRST_SENTENCE_IDX}: {first_selected['sentence']}")
print(f"AMR {FIRST_SENTENCE_IDX}:\n{first_selected['amr']}\n")
print(f"Sentence {SECOND_SENTENCE_IDX}: {second_selected['sentence']}")
print(f"AMR {SECOND_SENTENCE_IDX}:\n{second_selected['amr']}\n")
print(f"Matching triples: {details['match_count']}")
print(f"First graph triples: {details['first_count']}")
print(f"Second graph triples: {details['second_count']}")
print(f"Precision: {details['precision']:.4f}")
print(f"Recall: {details['recall']:.4f}")
print(f"SMATCH F-score: {details['f_score']:.4f}")
display(pd.DataFrame(details['matching_triples']))